In [16]:
import pandas as pd
import os
os.chdir("..")
from app.load_claims import load_claims

In [20]:
df = load_claims()
display(df.head())
print(df.shape)
print(df.dtypes)

/Users/electricalman/Desktop/internship_prep/gcp-intership-prep/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,claim_id,provider_id,payer_id,patient_age_bucket,service_line,procedure_group,claim_amount,allowed_amount,patient_responsibility,days_to_submit,prior_auth_required,prior_auth_present,eligibility_verified,submitted_at,claim_status,was_denied
0,C34,PROV_15,PAYER_5,adult,imaging,diagnostic,1358.0,1134.0,452.0,58,1,0,0,2026-04-21,paid,0
1,C172,PROV_13,PAYER_3,adult,surgery,complex,6464.0,5412.0,146.0,34,1,0,1,2025-12-04,denied,1
2,C220,PROV_1,PAYER_1,adult,surgery,complex,8240.0,6900.0,70.0,10,1,0,1,2025-10-17,denied,1
3,C297,PROV_18,PAYER_3,senior,primary_care,routine,2089.0,2287.0,371.0,9,0,1,1,2025-08-01,paid,0
4,C403,PROV_4,PAYER_4,adult,emergency,urgent,6011.0,5573.0,349.0,31,0,1,1,2026-04-17,paid,0


(50, 16)
claim_id                   object
provider_id                object
payer_id                   object
patient_age_bucket         object
service_line               object
procedure_group            object
claim_amount              float64
allowed_amount            float64
patient_responsibility    float64
days_to_submit              Int64
prior_auth_required         Int64
prior_auth_present          Int64
eligibility_verified        Int64
submitted_at               dbdate
claim_status               object
was_denied                  Int64
dtype: object


In [ ]:
print(f"The denial rate is: {df['was_denied'].mean()}")

The denial rate is: 0.42


This helps me find the payers that have highe denial rate

In [11]:
df.groupby("payer_id")["was_denied"].mean()

payer_id
PAYER_1    0.727273
PAYER_2    0.545455
PAYER_3    0.636364
PAYER_4         0.0
PAYER_5         0.0
Name: was_denied, dtype: Float64

In [13]:
df.groupby("prior_auth_required")["was_denied"].mean()

prior_auth_required
0    0.285714
1    0.517241
Name: was_denied, dtype: Float64

Service line denial rates

In [21]:
df.groupby("service_line")["was_denied"].mean()

service_line
emergency       0.214286
imaging         0.583333
primary_care    0.428571
surgery         0.470588
Name: was_denied, dtype: Float64

In [22]:
df.groupby("claim_status")["was_denied"].count()

claim_status
denied    21
paid      29
Name: was_denied, dtype: Int64

This checks helps me verify columns with potential data leakage

In [15]:
df.groupby("claim_status")["was_denied"].count()

claim_status
denied    21
paid      29
Name: was_denied, dtype: Int64

The target is not severely imbalanced. About 58% of claims were not denied and 42% were denied, so the model will have a reasonable number of examples from both classes. Accuracy is still not enough by itself, but imbalance is not a major issue in this synthetic dataset.

If it were 95% not denied and 5% denied, imbalance would be a huge problem.
At 58/42, the target is close enough to balanced for a baseline model.

In [23]:
df["was_denied"].value_counts(normalize=True)

was_denied
0    0.58
1    0.42
Name: proportion, dtype: Float64

In [35]:
df["claim_amount"].min()

np.float64(267.0)

In [36]:
df["claim_amount"].max()


np.float64(9017.0)

In [37]:
(df["claim_amount"] > 100).sum()

np.int64(50)

1. Are high claim amounts denied more often?

In [40]:
df["claim_bucket_bucket"] = pd.cut(
    df["claim_amount"],
    bins=[0,500,5000,7500, float("inf")],
    labels=["low","medium", "high", "very_high"]
)

df.groupby("claim_bucket_bucket")["was_denied"].mean()

/var/folders/5s/wp7dyzq57dvbmqdyw5535vtm0000gn/T/ipykernel_97230/2163954702.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("claim_bucket_bucket")["was_denied"].mean()


claim_bucket_bucket
low          0.333333
medium           0.36
high         0.571429
very_high       0.375
Name: was_denied, dtype: Float64

2. Does missing prior authorization increase denial rate?

In [44]:
df.groupby(["prior_auth_required", "prior_auth_present"])["was_denied"].agg(["count", "mean"]).reset_index()

,prior_auth_required,prior_auth_present,count,mean
0,0,0,6,0.333333
1,0,1,15,0.266667
2,1,0,9,0.555556
3,1,1,20,0.5


3. Does lack of eligibility verification increase denial rate?

In [46]:
df.groupby("eligibility_verified")["was_denied"].agg(["count", "mean"]).reset_index()

,eligibility_verified,count,mean
0,0,9,0.0
1,1,41,0.512195


In [47]:
df.groupby(["claim_status", "was_denied"]).size().reset_index(name="count")

,claim_status,was_denied,count
0,denied,1,21
1,paid,0,29
